# 11 — Optimizers and Gradient Descent

In the previous notebook, we learned how loss functions measure prediction error.

Now we will study how PyTorch uses gradients to update model parameters and reduce that loss.

This process is called:

> **Optimization**

An optimizer decides how model parameters such as weights and biases should change after gradients have been computed.

## In this notebook, we will learn:

1. Gradient descent intuition
2. Learning rate
3. Parameter updates
4. Batch, stochastic, and mini-batch gradient descent
5. Stochastic Gradient Descent
6. `torch.optim.SGD`
7. Momentum
8. Adam
9. `torch.optim.Adam`
10. `optimizer.zero_grad()`
11. `loss.backward()`
12. `optimizer.step()`
13. Weight decay
14. Comparing SGD and Adam
15. Learning-rate effects
16. Parameter groups
17. Inspecting optimizer state
18. Common optimizer mistakes
19. Building a clean optimization loop
20. Practice exercises

## Main Goal

By the end of this notebook, you should understand this training step:

$$
\boxed{
\text{zero gradients}
\rightarrow
\text{forward pass}
\rightarrow
\text{loss}
\rightarrow
\text{backward pass}
\rightarrow
\text{optimizer step}
}
$$

The most important idea is:

> **Gradients tell us the direction of change. The optimizer decides how to use those gradients to update parameters.**


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. What Is Optimization?

Training a neural network means finding parameter values that make the loss small.

Suppose a model has parameters:

$$
\theta
$$

and loss:

$$
L(\theta)
$$

Training tries to find parameter values that minimize:

$$
\boxed{L(\theta)}
$$

The optimizer repeatedly changes the parameters based on their gradients.


# 2. Gradient Descent Intuition

Suppose a parameter is called:

$$
w
$$

and the loss is:

$$
L(w)
$$

The derivative:

$$
\frac{dL}{dw}
$$

tells us how the loss changes when $w$ changes.

If the gradient is positive, increasing $w$ locally increases the loss.

If the gradient is negative, increasing $w$ locally decreases the loss.

To reduce the loss, gradient descent moves in the opposite direction of the gradient.


# 3. The Gradient Descent Update Rule

The basic update rule is:

$$
\boxed{
w_{new}
=
w_{old}
-
\eta
\frac{dL}{dw}
}
$$

where:

$$
\eta
$$

is the learning rate.

For a vector of parameters:

$$
\boxed{
\theta_{new}
=
\theta_{old}
-
\eta
\nabla_{\theta}L
}
$$


# 4. Simple Gradient Descent Example

Consider:

$$
L(w)=(w-4)^2
$$

The minimum occurs at:

$$
\boxed{w=4}
$$

The derivative is:

$$
\frac{dL}{dw}=2(w-4)
$$

Suppose:

$$
w=0
$$

Then:

$$
\frac{dL}{dw}=2(0-4)=-8
$$

With learning rate:

$$
\eta=0.1
$$

the update is:

$$
w_{new}
=
0-0.1(-8)
=
0.8
$$

So the parameter moves toward `4`.


In [ ]:
w = torch.tensor(0.0)

learning_rate = 0.1

gradient = 2 * (w - 4)

w_new = w - learning_rate * gradient

print("Old w:", w.item())
print("Gradient:", gradient.item())
print("New w:", w_new.item())


# 5. Gradient Descent With Autograd

Instead of deriving the gradient manually, PyTorch can compute it.


In [ ]:
w = torch.tensor(0.0, requires_grad=True)

loss = (w - 4) ** 2

loss.backward()

print("Loss:", loss.item())
print("Gradient:", w.grad.item())


Now perform the update.

We use:

`torch.no_grad()`

because the parameter update itself should not become part of the computational graph.


In [ ]:
learning_rate = 0.1

with torch.no_grad():
    w -= learning_rate * w.grad

print("Updated w:", w.item())


After the update, clear the gradient.


In [ ]:
w.grad.zero_()

print("Gradient after clearing:", w.grad)


# 6. Repeating Gradient Descent

Training requires many update steps.


In [ ]:
w = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.1
steps = 20

manual_history = []

for step in range(steps):
    loss = (w - 4) ** 2

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad

    w.grad.zero_()

    manual_history.append(
        (step + 1, w.item(), loss.item())
    )

    print(
        f"Step {step + 1:02d} | "
        f"w = {w.item():.4f} | "
        f"loss = {loss.item():.6f}"
    )


The parameter should move closer and closer to:

$$
\boxed{4}
$$

while the loss approaches zero.


# 7. What Is the Learning Rate?

The learning rate controls the size of each parameter update.

The update rule is:

$$
\theta_{new}
=
\theta_{old}
-
\eta\nabla L
$$

The learning rate is:

$$
\eta
$$

It is one of the most important hyperparameters in deep learning.


# 8. Learning Rate Too Small

If the learning rate is extremely small:

- Updates are tiny
- Training progresses slowly
- Many iterations may be needed

Example:

$$
\eta=0.001
$$

may move toward the minimum very slowly.


# 9. Learning Rate Too Large

If the learning rate is too large:

- Updates can overshoot the minimum
- Loss may oscillate
- Loss may increase
- Training can become unstable
- Values can sometimes become `inf` or `nan`

So:

> **A larger learning rate is not automatically better.**


# 10. Comparing Learning Rates

Let's optimize the same simple loss using different learning rates.


In [ ]:
def optimize_scalar(lr, steps=25):
    w = torch.tensor(0.0, requires_grad=True)
    losses = []

    for _ in range(steps):
        loss = (w - 4) ** 2

        loss.backward()

        with torch.no_grad():
            w -= lr * w.grad

        w.grad.zero_()

        losses.append(loss.item())

    return losses, w.item()

loss_small, w_small = optimize_scalar(
    lr=0.01
)

loss_good, w_good = optimize_scalar(
    lr=0.1
)

loss_large, w_large = optimize_scalar(
    lr=1.1
)

print("Small LR final w:", w_small)
print("Good LR final w:", w_good)
print("Large LR final w:", w_large)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_small, label="lr = 0.01")
plt.plot(loss_good, label="lr = 0.1")
plt.plot(loss_large, label="lr = 1.1")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Learning Rate Comparison")
plt.yscale("log")
plt.legend()
plt.show()


# 11. Batch, Stochastic, and Mini-Batch Gradient Descent

The phrase **Stochastic Gradient Descent** can be used broadly, but it is useful to understand three data-usage patterns.

## Batch Gradient Descent

Uses the entire training dataset to compute one gradient update.

## Stochastic Gradient Descent

Uses one training example per update.

## Mini-Batch Gradient Descent

Uses a small batch of examples per update.

Modern deep learning usually uses:

> **Mini-batch optimization**

even when the optimizer is called `SGD`.


# 12. Why Mini-Batches Are Common

Mini-batches provide a useful compromise.

Compared with one sample at a time, they:

- Use hardware efficiently
- Produce less noisy gradient estimates

Compared with the full dataset at once, they:

- Require less memory
- Allow more frequent updates
- Scale better to large datasets

Later, `DataLoader` will help us create mini-batches.


# 13. PyTorch Optimizers

PyTorch provides optimizers inside:

`torch.optim`

Common examples:

- `torch.optim.SGD`
- `torch.optim.Adam`
- `torch.optim.AdamW`
- `torch.optim.RMSprop`

In this notebook, we will focus on:

- SGD
- SGD with momentum
- Adam


In [ ]:
import torch.optim as optim

print(optim.SGD)
print(optim.Adam)


# 14. Why Use an Optimizer Object?

Without an optimizer, we might update parameters manually:

```python
with torch.no_grad():
    parameter -= learning_rate * parameter.grad
```

An optimizer automates parameter updates.

Typical usage:

```python
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)
```

Then:

```python
optimizer.step()
```

updates all registered parameters that have gradients.


# 15. A Small Regression Dataset

We will create:

$$
y=3x+2+\text{noise}
$$

and train a small linear model.


In [ ]:
torch.manual_seed(42)

x = torch.linspace(
    -5,
    5,
    100
).unsqueeze(1)

noise = torch.randn_like(x) * 1.0

y = 3 * x + 2 + noise

print("x shape:", x.shape)
print("y shape:", y.shape)


# 16. The Model and Loss

We will use:

`nn.Linear(1,1)`

and:

`nn.MSELoss()`


In [ ]:
torch.manual_seed(42)

model = nn.Linear(
    in_features=1,
    out_features=1
)

criterion = nn.MSELoss()

print(model)


# 17. Creating `torch.optim.SGD`

Pass the model parameters and learning rate to the optimizer.


In [ ]:
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)

print(optimizer)


# 18. The Standard Optimization Step

A standard PyTorch optimization step usually contains:

1. `optimizer.zero_grad()`
2. `predictions = model(inputs)`
3. `loss = criterion(predictions, targets)`
4. `loss.backward()`
5. `optimizer.step()`

Each line has a different purpose.


# 19. `optimizer.zero_grad()`

PyTorch gradients accumulate by default.

Therefore, before computing gradients for the next training step, we normally clear old gradients:

```python
optimizer.zero_grad()
```

Without this, the new gradients are added to previous gradients.


In [ ]:
optimizer.zero_grad()

for name, parameter in model.named_parameters():
    print(name, parameter.grad)


# 20. `loss.backward()`

`loss.backward()` asks Autograd to compute gradients of the loss with respect to tracked parameters.

Afterward:

```python
parameter.grad
```

contains the gradient.


In [ ]:
optimizer.zero_grad()

predictions = model(x)

loss = criterion(
    predictions,
    y
)

loss.backward()

print("Loss:", loss.item())

for name, parameter in model.named_parameters():
    print(
        name,
        "| grad shape:",
        parameter.grad.shape
    )


# 21. `optimizer.step()`

`optimizer.step()` uses the gradients currently stored in the model parameters and updates the parameters according to the optimizer's rule.


In [ ]:
before_weight = model.weight.detach().clone()
before_bias = model.bias.detach().clone()

optimizer.step()

after_weight = model.weight.detach().clone()
after_bias = model.bias.detach().clone()

print("Weight before:", before_weight)
print("Weight after :", after_weight)

print()

print("Bias before:", before_bias)
print("Bias after :", after_bias)


# 22. Complete SGD Training Loop

Now combine everything.


In [ ]:
torch.manual_seed(42)

sgd_model = nn.Linear(1, 1)

criterion = nn.MSELoss()

sgd_optimizer = optim.SGD(
    sgd_model.parameters(),
    lr=0.01
)

epochs = 200
sgd_losses = []

for epoch in range(epochs):
    sgd_optimizer.zero_grad()

    predictions = sgd_model(x)

    loss = criterion(
        predictions,
        y
    )

    loss.backward()

    sgd_optimizer.step()

    sgd_losses.append(
        loss.item()
    )

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Loss: {loss.item():.4f}"
        )


# 23. Inspecting Learned Parameters

The true relationship is approximately:

$$
y=3x+2
$$

Let's inspect what SGD learned.


In [ ]:
print(
    "Learned weight:",
    sgd_model.weight.item()
)

print(
    "Learned bias:",
    sgd_model.bias.item()
)


# 24. Visualizing SGD Training


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sgd_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("SGD Training Loss")
plt.show()


# 25. What Is Momentum?

Plain SGD uses the current gradient to update parameters.

Momentum adds a running influence from previous update directions.

Intuitively, it behaves like giving optimization some memory.

This can help:

- Accelerate movement in consistent directions
- Reduce some oscillation
- Improve optimization in certain landscapes

In PyTorch:

```python
torch.optim.SGD(
    parameters,
    lr=...,
    momentum=...
)
```


# 26. SGD With Momentum

A common momentum value is:

$$
0.9
$$

but it is a hyperparameter, not a universal rule.


In [ ]:
torch.manual_seed(42)

momentum_model = nn.Linear(1, 1)

momentum_optimizer = optim.SGD(
    momentum_model.parameters(),
    lr=0.01,
    momentum=0.9
)

momentum_losses = []

for epoch in range(200):
    momentum_optimizer.zero_grad()

    predictions = momentum_model(x)

    loss = criterion(
        predictions,
        y
    )

    loss.backward()

    momentum_optimizer.step()

    momentum_losses.append(
        loss.item()
    )

print(
    "Final momentum loss:",
    momentum_losses[-1]
)


# 27. Momentum State

Optimizers can maintain internal state.

For SGD with momentum, the optimizer maintains momentum-related buffers after updates begin.


In [ ]:
print(
    "Number of optimizer state entries:",
    len(momentum_optimizer.state)
)

for parameter, state in momentum_optimizer.state.items():
    print(
        "Parameter shape:",
        tuple(parameter.shape)
    )

    print(
        "State keys:",
        state.keys()
    )


# 28. What Is Adam?

Adam stands for:

> **Adaptive Moment Estimation**

Adam adapts the effective update for each parameter using running statistics of gradients.

Conceptually, Adam combines ideas related to:

- Momentum
- Per-parameter adaptive scaling

This often makes Adam a strong practical starting optimizer.


# 29. Creating `torch.optim.Adam`


In [ ]:
torch.manual_seed(42)

adam_model = nn.Linear(1, 1)

adam_optimizer = optim.Adam(
    adam_model.parameters(),
    lr=0.05
)

print(adam_optimizer)


# 30. Training With Adam

The training-loop structure does not change.

Only the optimizer changes.


In [ ]:
adam_losses = []

for epoch in range(200):
    adam_optimizer.zero_grad()

    predictions = adam_model(x)

    loss = criterion(
        predictions,
        y
    )

    loss.backward()

    adam_optimizer.step()

    adam_losses.append(
        loss.item()
    )

print(
    "Final Adam loss:",
    adam_losses[-1]
)

print(
    "Learned weight:",
    adam_model.weight.item()
)

print(
    "Learned bias:",
    adam_model.bias.item()
)


# 31. SGD vs Adam

Both optimizers use gradients, but their update strategies differ.

$$
\begin{array}{|c|c|}
\hline
\textbf{SGD} & \textbf{Adam} \\
\hline
\text{Simple update rule} & \text{Adaptive update rule} \\
\hline
\text{Few hyperparameters} & \text{Maintains optimizer state} \\
\hline
\text{Often benefits from tuning} & \text{Often works well quickly} \\
\hline
\text{Common in many vision settings} & \text{Common across many tasks} \\
\hline
\end{array}
$$

There is no optimizer that is always best.

The best optimizer depends on:

- Architecture
- Data
- Learning rate
- Regularization
- Training schedule
- Evaluation objective


# 32. Comparing SGD and Adam Loss Curves


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    sgd_losses,
    label="SGD"
)
plt.plot(
    momentum_losses,
    label="SGD + Momentum"
)
plt.plot(
    adam_losses,
    label="Adam"
)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Optimizer Comparison")
plt.legend()
plt.show()


# 33. Be Careful When Comparing Optimizers

A fair optimizer comparison requires care.

Different optimizers may need different:

- Learning rates
- Weight decay values
- Schedules
- Number of epochs

So do not conclude:

> Adam is always better

or:

> SGD is always better

based on one toy run.

The correct comparison is task-dependent.


# 34. Adam Optimizer State

Adam maintains internal running statistics for trainable parameters.

After training:


In [ ]:
print(
    "Number of Adam state entries:",
    len(adam_optimizer.state)
)

for parameter, state in adam_optimizer.state.items():
    print(
        "Parameter shape:",
        tuple(parameter.shape)
    )

    print(
        "State keys:",
        state.keys()
    )

    break


# 35. Weight Decay

Weight decay is a regularization technique that discourages parameters from becoming unnecessarily large.

Optimizers support a:

`weight_decay`

argument.

Example:

```python
optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)
```

or:

```python
optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
```


# 36. Why Use Weight Decay?

Large model parameters can sometimes contribute to overfitting.

Weight decay adds pressure toward smaller parameter values.

It is one form of regularization.

Later, we will study regularization in more depth.


In [ ]:
weight_decay_model = nn.Linear(
    10,
    1
)

weight_decay_optimizer = optim.SGD(
    weight_decay_model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

print(weight_decay_optimizer)


# 37. Adam and AdamW

PyTorch also provides:

`torch.optim.AdamW`

AdamW uses **decoupled weight decay**.

For many modern deep-learning workflows, AdamW is a common choice when using Adam-style optimization with weight decay.

We will focus mainly on Adam in this notebook, but remember:

> **Adam and AdamW are not exactly the same optimizer.**


In [ ]:
adamw_model = nn.Linear(
    10,
    1
)

adamw_optimizer = optim.AdamW(
    adamw_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print(adamw_optimizer)


# 38. Optimizer Hyperparameters

Optimizers contain hyperparameters.

For SGD, important examples include:

- Learning rate
- Momentum
- Weight decay

For Adam, important examples include:

- Learning rate
- `betas`
- `eps`
- Weight decay

Most beginners should first learn the default behavior before aggressively changing every option.


# 39. Inspecting Optimizer Parameter Groups

Optimizers organize parameters into:

> **Parameter groups**

We can inspect them using:

`optimizer.param_groups`


In [ ]:
for index, group in enumerate(
    adam_optimizer.param_groups
):
    print(
        "Parameter group:",
        index
    )

    print(
        "Learning rate:",
        group["lr"]
    )

    print(
        "Number of tensors:",
        len(group["params"])
    )


# 40. Why Parameter Groups Matter

Parameter groups let us use different hyperparameters for different parts of a model.

For example:

- Lower learning rate for pretrained layers
- Higher learning rate for a new classifier
- Different weight decay settings

This becomes useful in transfer learning.


# 41. Different Learning Rates for Different Layers

Let's create a two-layer model.


In [ ]:
class TwoLayerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(
            10,
            20
        )

        self.layer2 = nn.Linear(
            20,
            2
        )

    def forward(self, x):
        x = torch.relu(
            self.layer1(x)
        )

        return self.layer2(x)

two_layer_model = TwoLayerModel()


We can create different parameter groups.


In [ ]:
group_optimizer = optim.SGD(
    [
        {
            "params":
                two_layer_model.layer1.parameters(),
            "lr": 0.001
        },
        {
            "params":
                two_layer_model.layer2.parameters(),
            "lr": 0.01
        }
    ]
)

for index, group in enumerate(
    group_optimizer.param_groups
):
    print(
        f"Group {index} learning rate:",
        group["lr"]
    )


# 42. Changing the Learning Rate During Training

The learning rate can be changed by modifying optimizer parameter groups.

Example:


In [ ]:
optimizer_example = optim.SGD(
    two_layer_model.parameters(),
    lr=0.01
)

print(
    "Before:",
    optimizer_example.param_groups[0]["lr"]
)

optimizer_example.param_groups[0]["lr"] = 0.001

print(
    "After:",
    optimizer_example.param_groups[0]["lr"]
)


Later, we will study **learning-rate schedulers**, which automate learning-rate changes during training.


# 43. Order of Operations

A clean training step usually looks like:

```python
optimizer.zero_grad()

predictions = model(inputs)

loss = criterion(
    predictions,
    targets
)

loss.backward()

optimizer.step()
```

The order matters.


# 44. Why `zero_grad()` Comes Before the New Backward Pass

Because gradients accumulate:

```python
loss.backward()
```

adds new gradients to existing `.grad` values.

So clearing old gradients before the new backward pass gives us gradients from the current step only.


# 45. Can `zero_grad()` Come After `step()`?

You may see loops where gradients are cleared at the end of an iteration instead.

That can work if done consistently.

However, a very clear beginner pattern is:

```python
optimizer.zero_grad()
forward
loss
backward
optimizer.step()
```

because each iteration begins by explicitly clearing old gradients.


# 46. `zero_grad(set_to_none=True)`

PyTorch supports:

```python
optimizer.zero_grad(
    set_to_none=True
)
```

This sets gradients to `None` instead of filling them with zeros.

This can have performance and memory benefits.

The default behavior can vary by PyTorch API/version details, so the important conceptual rule is:

> **Clear old gradients before the next gradient computation when you do not want accumulation.**


In [ ]:
temp_model = nn.Linear(
    2,
    1
)

temp_optimizer = optim.SGD(
    temp_model.parameters(),
    lr=0.01
)

temp_optimizer.zero_grad(
    set_to_none=True
)

for name, parameter in temp_model.named_parameters():
    print(
        name,
        parameter.grad
    )


# 47. What Happens If We Forget `optimizer.step()`?

If we do:

```python
loss.backward()
```

but never call:

```python
optimizer.step()
```

gradients are computed, but parameters do not update.

Therefore the model will not learn from those gradients.


In [ ]:
demo_model = nn.Linear(
    1,
    1
)

demo_optimizer = optim.SGD(
    demo_model.parameters(),
    lr=0.1
)

demo_x = torch.tensor(
    [[1.0]]
)

demo_y = torch.tensor(
    [[5.0]]
)

before = demo_model.weight.detach().clone()

demo_optimizer.zero_grad()

demo_loss = nn.MSELoss()(
    demo_model(demo_x),
    demo_y
)

demo_loss.backward()

after_backward = (
    demo_model.weight.detach().clone()
)

print(
    "Changed after backward only:",
    not torch.equal(
        before,
        after_backward
    )
)


Now call `step()`.


In [ ]:
demo_optimizer.step()

after_step = (
    demo_model.weight.detach().clone()
)

print(
    "Changed after optimizer.step():",
    not torch.equal(
        before,
        after_step
    )
)


# 48. What Happens If We Forget `loss.backward()`?

If we skip:

`loss.backward()`

the optimizer has no newly computed gradients to use.

In normal training:

> **Backward computes gradients. Step uses them.**


# 49. What Happens If We Forget `zero_grad()`?

Then gradients accumulate across iterations.

This is sometimes intentional.

For example, gradient accumulation can simulate a larger effective batch size.

But if you do not intend to accumulate gradients, forgetting `zero_grad()` changes training behavior.


# 50. Intentional Gradient Accumulation

Suppose memory only allows small batches.

We may intentionally accumulate gradients across several mini-batches before calling:

`optimizer.step()`

Conceptually:

```python
optimizer.zero_grad()

for several mini_batches:
    loss = ...
    loss.backward()

optimizer.step()
```

In practice, the loss is often scaled appropriately when accumulating.

We will study full training-loop design later.


# 51. A Clean Reusable Training Step

Let's write a function for one optimization step.


In [ ]:
def training_step(
    model,
    inputs,
    targets,
    criterion,
    optimizer
):
    model.train()

    optimizer.zero_grad()

    predictions = model(inputs)

    loss = criterion(
        predictions,
        targets
    )

    loss.backward()

    optimizer.step()

    return loss.item()


# 52. Testing the Training-Step Function


In [ ]:
torch.manual_seed(42)

clean_model = nn.Linear(
    1,
    1
)

clean_optimizer = optim.SGD(
    clean_model.parameters(),
    lr=0.01
)

clean_criterion = nn.MSELoss()

loss_value = training_step(
    clean_model,
    x,
    y,
    clean_criterion,
    clean_optimizer
)

print(
    "Training loss:",
    loss_value
)


# 53. Building a Clean Training Loop


In [ ]:
torch.manual_seed(42)

clean_model = nn.Linear(
    1,
    1
)

clean_optimizer = optim.Adam(
    clean_model.parameters(),
    lr=0.05
)

clean_criterion = nn.MSELoss()

clean_losses = []

for epoch in range(100):
    loss_value = training_step(
        clean_model,
        x,
        y,
        clean_criterion,
        clean_optimizer
    )

    clean_losses.append(
        loss_value
    )

print(
    "Final loss:",
    clean_losses[-1]
)


# 54. Evaluation Is Different From Optimization

During validation or testing:

- Do not call `optimizer.step()`
- Do not update parameters
- Usually do not need gradients

Typical pattern:

```python
model.eval()

with torch.no_grad():
    predictions = model(inputs)
    loss = criterion(
        predictions,
        targets
    )
```


In [ ]:
clean_model.eval()

with torch.no_grad():
    predictions = clean_model(x)

    evaluation_loss = clean_criterion(
        predictions,
        y
    )

print(
    "Evaluation loss:",
    evaluation_loss.item()
)


# 55. Optimizer Does Not Replace Autograd

An optimizer does not calculate gradients by itself.

The roles are:

$$
\begin{array}{|c|c|}
\hline
\textbf{Component} & \textbf{Responsibility} \\
\hline
Autograd & \text{Computes gradients} \\
\hline
Optimizer & \text{Updates parameters} \\
\hline
Loss Function & \text{Defines objective} \\
\hline
Model & \text{Produces predictions} \\
\hline
\end{array}
$$

This separation is fundamental.


# 56. Common Optimizer Mistake — Wrong Learning Rate

Symptoms may include:

- Loss barely changes
- Loss explodes
- Loss oscillates
- `nan` values appear

The learning rate should be one of the first hyperparameters you inspect.


# 57. Common Optimizer Mistake — Optimizer Created With Wrong Parameters

If the optimizer does not receive the parameters you intend to train, those parameters will not be updated.

Typical pattern:

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=...
)
```


In [ ]:
sample_model = nn.Linear(
    3,
    2
)

sample_optimizer = optim.Adam(
    sample_model.parameters(),
    lr=0.001
)

number_of_optimizer_tensors = sum(
    len(group["params"])
    for group in sample_optimizer.param_groups
)

print(
    "Number of parameter tensors:",
    number_of_optimizer_tensors
)


# 58. Common Optimizer Mistake — Replacing Parameters After Creating the Optimizer

Optimizers hold references to parameter objects.

If you replace model parameters after constructing the optimizer, the optimizer may still reference the old parameter objects.

A safe rule:

> **Finish constructing the trainable model before creating the optimizer.**

If you change which parameters should be optimized, verify the optimizer parameter groups.


# 59. Common Optimizer Mistake — Forgetting Frozen Parameters

Parameters with:

`requires_grad=False`

will not receive normal gradients.

This is useful for freezing layers, but can be confusing if done accidentally.


In [ ]:
freeze_demo = TwoLayerModel()

for parameter in freeze_demo.layer1.parameters():
    parameter.requires_grad = False

for name, parameter in freeze_demo.named_parameters():
    print(
        name,
        "| trainable:",
        parameter.requires_grad
    )


# 60. Common Optimizer Mistake — Comparing Optimizers Unfairly

Do not compare:

- SGD at one arbitrary learning rate
- Adam at another arbitrary learning rate

and conclude one optimizer is universally superior.

A fair comparison should consider tuning each optimizer appropriately.


# 61. Optimizer Debugging Checklist

If training is not improving, inspect:

1. Is the loss finite?
2. Is the learning rate reasonable?
3. Are gradients present?
4. Are gradients finite?
5. Are parameters changing?
6. Did you call `optimizer.zero_grad()`?
7. Did you call `loss.backward()`?
8. Did you call `optimizer.step()`?
9. Does the optimizer contain the intended parameters?
10. Are any important layers frozen?
11. Is the loss function correct?
12. Are input and target shapes correct?


In [ ]:
debug_model = nn.Linear(
    2,
    1
)

debug_optimizer = optim.SGD(
    debug_model.parameters(),
    lr=0.01
)

debug_x = torch.randn(
    4,
    2
)

debug_y = torch.randn(
    4,
    1
)

debug_optimizer.zero_grad()

debug_predictions = debug_model(
    debug_x
)

debug_loss = nn.MSELoss()(
    debug_predictions,
    debug_y
)

debug_loss.backward()

print(
    "Loss:",
    debug_loss.item()
)

for name, parameter in debug_model.named_parameters():
    print(
        name,
        "| grad exists:",
        parameter.grad is not None,
        "| finite:",
        torch.isfinite(
            parameter.grad
        ).all().item()
    )


# 62. Checking Whether Parameters Actually Change

A powerful debugging technique is to compare a parameter before and after `optimizer.step()`.


In [ ]:
before = (
    debug_model.weight
    .detach()
    .clone()
)

debug_optimizer.step()

after = (
    debug_model.weight
    .detach()
    .clone()
)

print(
    "Weight changed:",
    not torch.equal(
        before,
        after
    )
)


# 63. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

For:

$$
L(w)=(w-5)^2
$$

find the gradient manually at:

$$
w=2
$$

## Exercise 2

Perform one gradient-descent update using:

$$
w=2
$$

$$
\eta=0.1
$$

## Exercise 3

Optimize:

$$
L(w)=(w-5)^2
$$

using Autograd for 20 steps.

## Exercise 4

Create:

`nn.Linear(3,1)`

with:

`torch.optim.SGD`

## Exercise 5

Write one complete training step using:

- `zero_grad()`
- Forward pass
- MSE loss
- `backward()`
- `step()`

## Exercise 6

Repeat Exercise 5 using Adam.

## Exercise 7

Train a simple regression model with:

- SGD
- SGD + momentum
- Adam

and compare their loss curves.

## Exercise 8

Create SGD with:

$$
momentum=0.9
$$

## Exercise 9

Create Adam with weight decay.

## Exercise 10

Freeze the first layer of a two-layer model and verify that it does not receive gradients.


# 64. Conceptual Challenges

Answer before running code.

## Challenge 1

What is the difference between:

`loss.backward()`

and:

`optimizer.step()`?

## Challenge 2

Why must old gradients usually be cleared?

## Challenge 3

What happens if the learning rate is extremely small?

## Challenge 4

What can happen if the learning rate is extremely large?

## Challenge 5

Why can momentum help SGD?

## Challenge 6

Why does Adam need optimizer state?

## Challenge 7

What is weight decay trying to encourage?

## Challenge 8

Why should the optimizer usually be created after the model architecture is finalized?

## Challenge 9

Why should validation code not call `optimizer.step()`?

## Challenge 10

What is the difference between optimization and evaluation?


# 65. Exercise Solutions


In [ ]:
# Exercise 1
w_value = 2.0

gradient = 2 * (
    w_value - 5
)

print(
    "Exercise 1 gradient:",
    gradient
)

# Exercise 2
learning_rate = 0.1

new_w = (
    w_value
    - learning_rate
    * gradient
)

print(
    "Exercise 2 new w:",
    new_w
)

# Exercise 3
w = torch.tensor(
    2.0,
    requires_grad=True
)

for _ in range(20):
    loss = (w - 5) ** 2

    loss.backward()

    with torch.no_grad():
        w -= 0.1 * w.grad

    w.grad.zero_()

print(
    "Exercise 3 w:",
    w.item()
)

# Exercise 4
model_ex4 = nn.Linear(
    3,
    1
)

optimizer_ex4 = optim.SGD(
    model_ex4.parameters(),
    lr=0.01
)

print(
    "Exercise 4:",
    optimizer_ex4
)

# Exercise 5
X_ex5 = torch.randn(
    8,
    3
)

y_ex5 = torch.randn(
    8,
    1
)

criterion_ex5 = nn.MSELoss()

optimizer_ex4.zero_grad()

pred_ex5 = model_ex4(
    X_ex5
)

loss_ex5 = criterion_ex5(
    pred_ex5,
    y_ex5
)

loss_ex5.backward()

optimizer_ex4.step()

print(
    "Exercise 5 loss:",
    loss_ex5.item()
)

# Exercise 6
model_ex6 = nn.Linear(
    3,
    1
)

optimizer_ex6 = optim.Adam(
    model_ex6.parameters(),
    lr=0.001
)

optimizer_ex6.zero_grad()

pred_ex6 = model_ex6(
    X_ex5
)

loss_ex6 = criterion_ex5(
    pred_ex6,
    y_ex5
)

loss_ex6.backward()

optimizer_ex6.step()

print(
    "Exercise 6 loss:",
    loss_ex6.item()
)

# Exercise 8
model_ex8 = nn.Linear(
    3,
    1
)

optimizer_ex8 = optim.SGD(
    model_ex8.parameters(),
    lr=0.01,
    momentum=0.9
)

print(
    "Exercise 8:",
    optimizer_ex8
)

# Exercise 9
model_ex9 = nn.Linear(
    3,
    1
)

optimizer_ex9 = optim.Adam(
    model_ex9.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print(
    "Exercise 9:",
    optimizer_ex9
)

# Exercise 10
model_ex10 = TwoLayerModel()

for parameter in model_ex10.layer1.parameters():
    parameter.requires_grad = False

X_ex10 = torch.randn(
    4,
    10
)

y_ex10 = torch.randn(
    4,
    2
)

optimizer_ex10 = optim.SGD(
    filter(
        lambda p: p.requires_grad,
        model_ex10.parameters()
    ),
    lr=0.01
)

optimizer_ex10.zero_grad()

loss_ex10 = nn.MSELoss()(
    model_ex10(X_ex10),
    y_ex10
)

loss_ex10.backward()

print(
    "Exercise 10 layer1 weight grad:",
    model_ex10.layer1.weight.grad
)

print(
    "Exercise 10 layer2 weight grad exists:",
    model_ex10.layer2.weight.grad is not None
)


# 66. Key Takeaways

In this notebook, we learned:

- Gradient descent intuition
- Parameter update rules
- Learning rates
- Batch vs stochastic vs mini-batch updates
- `torch.optim`
- `torch.optim.SGD`
- Momentum
- Adam
- `torch.optim.Adam`
- `optimizer.zero_grad()`
- `loss.backward()`
- `optimizer.step()`
- Optimizer state
- Weight decay
- AdamW
- Parameter groups
- Different learning rates for different layers
- Intentional gradient accumulation
- Clean training steps
- Evaluation without optimization
- Common optimizer mistakes
- Optimizer debugging

The core training step is:

$$
\boxed{
optimizer.zero\_grad()
\rightarrow
forward
\rightarrow
loss
\rightarrow
loss.backward()
\rightarrow
optimizer.step()
}
$$

Remember:

> **Autograd computes gradients. The optimizer uses those gradients to update parameters.**


# 67. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is gradient descent?
2. Why do we move opposite to the gradient?
3. What does the learning rate control?
4. What happens if the learning rate is too small?
5. What can happen if it is too large?
6. What is mini-batch gradient descent?
7. What does `optimizer.zero_grad()` do?
8. What does `loss.backward()` do?
9. What does `optimizer.step()` do?
10. What is the difference between Autograd and an optimizer?
11. What is SGD?
12. What does momentum add to SGD?
13. What is Adam?
14. Why does Adam maintain optimizer state?
15. What is weight decay?
16. What is AdamW?
17. What are optimizer parameter groups?
18. Why might different layers use different learning rates?
19. Why should parameters normally not update during evaluation?
20. What should you inspect when a model's loss does not decrease?


# Next Notebook

# 12 — Dataset and DataLoader

In the next notebook, we will study:

- Why data pipelines matter
- `Dataset`
- `TensorDataset`
- Custom `Dataset`
- `__len__()`
- `__getitem__()`
- `DataLoader`
- Batch size
- Shuffling
- Iterating through batches
- Feature and target shapes
- Train/validation/test datasets
- Mini-batch training
- `num_workers`
- Reproducible loading
- Common data-pipeline mistakes
